In [24]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [25]:
df= pd.read_csv('loan_data.csv', encoding = 'latin1')
df.head(3)

,loan_id,age,education,proof_submitted,loan_amount,asset_cost,no_of_loans,no_of_curr_loans,last_delinq_none,loan_default
0,1,27,1.0,Passport,504264,820920,2,2,0,0
1,2,48,1.0,Passport,728556,831444,6,2,0,0
2,3,30,2.0,GovID,642936,826092,0,0,0,1


In [26]:
df.dtypes

loan_id               int64
age                   int64
education           float64
proof_submitted      object
loan_amount           int64
asset_cost            int64
no_of_loans           int64
no_of_curr_loans      int64
last_delinq_none      int64
loan_default          int64
dtype: object

In [27]:
#to find 'unique' values of a specific column
df['proof_submitted'].unique()

array(['Passport', 'GovID', 'Driving', 'Age Card'], dtype=object)

In [28]:
df['education'].unique()

array([ 1.,  2., nan])

In [29]:
df.isnull().sum()

loan_id               0
age                   0
education           245
proof_submitted       0
loan_amount           0
asset_cost            0
no_of_loans           0
no_of_curr_loans      0
last_delinq_none      0
loan_default          0
dtype: int64

In [30]:
#to replace any NaN with '0'
df['education'].fillna(0, inplace=True)

# Check the null values again
df.isnull().sum()

C:\Users\Tim\AppData\Local\Temp\ipykernel_9088\177917620.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['education'].fillna(0, inplace=True)


loan_id             0
age                 0
education           0
proof_submitted     0
loan_amount         0
asset_cost          0
no_of_loans         0
no_of_curr_loans    0
last_delinq_none    0
loan_default        0
dtype: int64

In [31]:
df['education'] = df['education'].astype(int)
df.dtypes

loan_id              int64
age                  int64
education            int32
proof_submitted     object
loan_amount          int64
asset_cost           int64
no_of_loans          int64
no_of_curr_loans     int64
last_delinq_none     int64
loan_default         int64
dtype: object

In [32]:
#Grouping the loan defaults, 1. 'count' the numbers,2. Use 'len' (method to go to the end of the dataframe)
df.groupby("loan_default").count()/len(df)

,loan_id,age,education,proof_submitted,loan_amount,asset_cost,no_of_loans,no_of_curr_loans,last_delinq_none
loan_default,,,,,,,,,
0,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6
1,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4,0.4


In [33]:
#Create a function that assigns each of the values in the column (`1`,`2`, and `0`) 
#& apply it to the column
def get_education_category(education_value):
   if education_value == 1:
       return "Group_1"
   elif education_value == 2:
       return "Group_2"
   elif np.isnan(education_value):
       return "Group_3"

df["education"] = df["education"].apply(get_education_category)
df.head(3)

,loan_id,age,education,proof_submitted,loan_amount,asset_cost,no_of_loans,no_of_curr_loans,last_delinq_none,loan_default
0,1,27,Group_1,Passport,504264,820920,2,2,0,0
1,2,48,Group_1,Passport,728556,831444,6,2,0,0
2,3,30,Group_2,GovID,642936,826092,0,0,0,1


In [47]:
#df = pd.get_dummies(df,columns=['education','proof_submitted'])
#df.head(3)

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

X_train, X_test, y_train, y_test = train_test_split(
   df.drop(["loan_default"], axis=1),
   df["loan_default"],
   test_size=0.4,
)

In [40]:
from sklearn.model_selection import GridSearchCV

clf = GridSearchCV(
   RandomForestClassifier(),
   {
       "criterion": ["gini", "entropy"],
       "max_depth": [50 + i * 10 for i in range(10)],
       "oob_score": [True, False],
       "max_features": ["sqrt", None, 6],
       "min_impurity_decrease": [0.00, 0.001],
       "class_weight": ["balanced"],
   },
   scoring="recall",
   n_jobs=-1,
   verbose=3,
)

In [45]:
clf.fit(X_train, y_train)


Fitting 5 folds for each of 240 candidates, totalling 1200 fits


GridSearchCV(estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'class_weight': ['balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [50, 60, 70, 80, 90, 100, 110, 120, 130,
                                       140],
                         'max_features': ['sqrt', None, 6],
                         'min_impurity_decrease': [0.0, 0.001],
                         'oob_score': [True, False]},
             scoring='recall', verbose=3)

In [46]:
print("Recall score of best model:",
   round(recall_score(y_test, clf.predict(X_test)) * 100, 2),
   "%",
)
print("Best model parameters:")
clf.best_params_


Recall score of best model: 60.98 %
Best model parameters:


{'class_weight': 'balanced',
 'criterion': 'gini',
 'max_depth': 130,
 'max_features': 'sqrt',
 'min_impurity_decrease': 0.001,
 'oob_score': True}